# Best-Practice Clustering — A Real Project Template

This is the structure to copy for your own segmentation project. It shows:
1. **The sklearn `Pipeline`** — the #1 best practice (chains steps, prevents mistakes)
2. **The full lifecycle** — not just clustering, but what comes *after*: profile → name → act → predict new data

We use a synthetic **customer dataset** (age, income, spend score, visit frequency) because segmentation is the perfect 'what-after-clustering' teacher: the whole point is figuring out *who the groups are and what to do about them*.

**Swap in your own data** (a CSV you care about) and follow the same steps.

---

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

np.random.seed(0)

## Step 1 — Get the data

In a real project: `df = pd.read_csv('your_file.csv')`. Here we synthesize a believable customer table. The features are the columns you'd cluster on.

In [2]:
n = 400
df = pd.DataFrame({
    'age':         np.random.randint(18, 70, n),
    'income':      np.random.randint(15, 150, n) * 1000,
    'spend_score': np.random.randint(1, 100, n),     # 1-100, how much they spend
    'visit_freq':  np.random.poisson(5, n) + 1       # visits per month
})

print(df.shape)
df.head()

(400, 4)


,age,income,spend_score,visit_freq
0,62,89000,75,9
1,65,67000,68,4
2,18,66000,87,9
3,21,120000,61,4
4,21,33000,94,7


## Step 2 — ALWAYS explore before clustering

Best practice: never cluster blind. Check distributions, missing values, scales. This catches problems early.

In [3]:
print("Missing values:\n", df.isnull().sum().to_string())
print("\nScales (note how different they are):")
print(df.describe().loc[['min', 'mean', 'max']].round(0).T)
print("\n-> income is in tens of thousands, spend_score is 1-100. Must standardize.")

Missing values:
 age            0
income         0
spend_score    0
visit_freq     0

Scales (note how different they are):
                 min     mean       max
age             18.0     43.0      69.0
income       15000.0  88110.0  149000.0
spend_score      1.0     50.0      99.0
visit_freq       1.0      6.0      16.0

-> income is in tens of thousands, spend_score is 1-100. Must standardize.


## Step 3 — Build a PIPELINE (the key best practice)

Instead of separate `scaler.fit_transform`, `pca.fit_transform`, `kmeans.fit` calls with intermediate variables, chain them into ONE object. Benefits:
- **Order is locked in** — can't accidentally cluster before scaling.
- **One call** does everything: `pipe.fit_predict(df)`.
- **New data gets the exact same treatment** automatically — no re-deriving the scaler.
- **No data leakage** — the scaler/PCA are fit only on training data.

A Pipeline is just a list of `(name, step)` tuples. Each step transforms and passes to the next.

In [4]:
pipe = Pipeline([
    ('scale', StandardScaler()),          # step 1: standardize
    ('pca',   PCA(n_components=0.95)),     # step 2: keep 95% variance
    ('kmeans', KMeans(n_clusters=4, n_init=10, random_state=0))  # step 3: cluster
])

# Choose k properly first — sweep k THROUGH the pipeline
for k in range(2, 8):
    pipe.set_params(kmeans__n_clusters=k)        # note the double-underscore: step__param
    labels_k = pipe.fit_predict(df)
    # silhouette on the transformed data (after scale+pca)
    X_trans = pipe[:-1].transform(df)            # run all steps EXCEPT the last (clustering)
    print(f"k={k}: silhouette={silhouette_score(X_trans, labels_k):.3f}")

k=2: silhouette=0.196
k=3: silhouette=0.193
k=4: silhouette=0.207
k=5: silhouette=0.222
k=6: silhouette=0.220
k=7: silhouette=0.233


Notice two pipeline tricks:
- `pipe.set_params(kmeans__n_clusters=k)` — change a parameter of a named step using `stepname__paramname` (double underscore). This is how you tune steps inside a pipeline.
- `pipe[:-1].transform(df)` — slice the pipeline to run all steps *except* the last. Here it gives the scaled+PCA'd data so we can score the silhouette (which needs the data the clustering actually saw).

In [5]:
# Lock in the chosen k and do the final fit
BEST_K = 4
pipe.set_params(kmeans__n_clusters=BEST_K)
df['cluster'] = pipe.fit_predict(df)        # attach the cluster label to each customer

print(f"Final clustering with k={BEST_K}")
print(f"PCA kept {pipe.named_steps['pca'].n_components_} components")
print(f"Cluster sizes:\n{df['cluster'].value_counts().sort_index().to_string()}")

Final clustering with k=4
PCA kept 4 components
Cluster sizes:
cluster
0     96
1     67
2    130
3    107


---

# AFTER CLUSTERING — this is the part you wanted to learn

Clustering gave each customer a group number. That number is meaningless until we *interpret* it. Here's the real after-work.

## After-step 1 — PROFILE the clusters (what makes each group different?)

Go back to the ORIGINAL features and average them within each cluster. This is the single most important after-clustering action — it turns 'cluster 0' into a description.

In [6]:
# Average each original feature within each cluster
profile = df.groupby('cluster')[['age', 'income', 'spend_score', 'visit_freq']].mean().round(1)
print(profile)
print("\nEach row is a customer 'persona'. Read across to see what defines the group.")

          age    income  spend_score  visit_freq
cluster                                         
0        50.7   43708.3         41.4         5.3
1        40.9   83089.6         59.2         9.7
2        27.6   95130.8         51.5         5.0
3        55.1  122560.7         50.2         5.5

Each row is a customer 'persona'. Read across to see what defines the group.


In [7]:
# Visualize the profiles as a heatmap — easy to compare groups at a glance
# Standardize columns so colors are comparable across features with different scales
prof_z = (profile - profile.mean()) / profile.std()
fig = px.imshow(prof_z.T, text_auto='.1f', color_continuous_scale='RdBu_r', aspect='auto',
    labels=dict(x='cluster', y='feature', color='above/below avg'))
fig.update_layout(title='Cluster profiles: red = high for that feature, blue = low',
    width=650, height=380)
fig.show()
print("Red cells = this group scores HIGH on that feature; blue = LOW. Read each column as a persona.")

Red cells = this group scores HIGH on that feature; blue = LOW. Read each column as a persona.


## After-step 2 — NAME the segments (translate numbers to humans)

Look at each profile and give it a meaningful name. This is judgment, not code — it's where domain knowledge enters. Example logic (yours will differ based on your actual profile numbers):

In [8]:
# Build names from the profile (this logic is illustrative — adapt to YOUR numbers)
names = {}
for c in profile.index:
    row = profile.loc[c]
    inc = 'high-income' if row['income'] > profile['income'].median() else 'budget'
    spend = 'big-spender' if row['spend_score'] > profile['spend_score'].median() else 'low-spend'
    age = 'young' if row['age'] < profile['age'].median() else 'older'
    names[c] = f"{age}, {inc}, {spend}"

df['segment'] = df['cluster'].map(names)
for c, nm in names.items():
    print(f"cluster {c}: {nm}  ({(df['cluster']==c).sum()} customers)")

cluster 0: older, budget, low-spend  (96 customers)
cluster 1: young, budget, big-spender  (67 customers)
cluster 2: young, high-income, big-spender  (130 customers)
cluster 3: older, high-income, low-spend  (107 customers)


## After-step 3 — ACT on the segments (the business decision)

The whole reason you clustered. Each segment gets a different action. In code this might be tagging records for a campaign; the point is the cluster label now *drives a decision*.

In [9]:
# Example: assign a marketing action per segment
actions = {}
for c in profile.index:
    row = profile.loc[c]
    if row['spend_score'] > 55 and row['income'] > 80000:
        actions[c] = 'VIP loyalty program'
    elif row['spend_score'] < 40:
        actions[c] = 're-engagement discount'
    else:
        actions[c] = 'standard newsletter'

df['action'] = df['cluster'].map(actions)
print("Action plan per segment:")
print(df.groupby('cluster')['action'].first().to_string())
print("\nNow every customer has a targeted action — the cluster label became a decision.")

Action plan per segment:
cluster
0    standard newsletter
1    VIP loyalty program
2    standard newsletter
3    standard newsletter

Now every customer has a targeted action — the cluster label became a decision.


## After-step 4 — PREDICT the segment for a NEW customer

Clustering only labeled existing data. When a new customer arrives, the fitted pipeline assigns them to the nearest cluster — applying the SAME scaling and PCA automatically. This is the payoff of using a Pipeline.

In [10]:
# A new customer walks in
new_customer = pd.DataFrame([{
    'age': 28, 'income': 95000, 'spend_score': 75, 'visit_freq': 8
}])

# The pipeline scales + PCAs + assigns cluster, all in one .predict()
pred_cluster = pipe.predict(new_customer)[0]
print(f"New customer assigned to cluster {pred_cluster}")
print(f"  Segment: {names[pred_cluster]}")
print(f"  Recommended action: {actions[pred_cluster]}")
print("\nNo manual scaling needed — the pipeline remembered the training transformations.")

New customer assigned to cluster 1
  Segment: young, budget, big-spender
  Recommended action: VIP loyalty program

No manual scaling needed — the pipeline remembered the training transformations.


## After-step 5 — VALIDATE that clusters are stable (don't trust blindly)

Best practice: check the clusters aren't a fluke. Re-run with different random seeds; if the grouping is stable, it's likely real structure. (Bootstrap, later in your course, formalizes this idea.)

In [11]:
from sklearn.metrics import adjusted_rand_score

base = pipe.fit_predict(df[['age','income','spend_score','visit_freq']])
print("Stability check — re-cluster with different seeds, compare to baseline:")
for seed in [1, 2, 3, 42]:
    p = Pipeline([
        ('scale', StandardScaler()),
        ('pca', PCA(0.95)),
        ('kmeans', KMeans(n_clusters=BEST_K, n_init=10, random_state=seed))
    ]).fit_predict(df[['age','income','spend_score','visit_freq']])
    print(f"  seed {seed}: agreement with baseline ARI={adjusted_rand_score(base, p):.3f}")
print("\nHigh ARI across seeds = stable, trustworthy clusters. Low = fragile, be skeptical.")
print("\nNOTE: on THIS synthetic random data the ARIs come out low/inconsistent —")
print("which is the honest signal that this fake data has no real cluster structure.")
print("On real data with genuine groups, you'd see high ARI across seeds. This is")
print("exactly the check that stops you from trusting clusters that aren't really there.")

Stability check — re-cluster with different seeds, compare to baseline:
  seed 1: agreement with baseline ARI=0.957
  seed 2: agreement with baseline ARI=0.288
  seed 3: agreement with baseline ARI=0.798
  seed 42: agreement with baseline ARI=0.838

High ARI across seeds = stable, trustworthy clusters. Low = fragile, be skeptical.

NOTE: on THIS synthetic random data the ARIs come out low/inconsistent —
which is the honest signal that this fake data has no real cluster structure.
On real data with genuine groups, you'd see high ARI across seeds. This is
exactly the check that stops you from trusting clusters that aren't really there.


---

# The complete best-practice template (your reference)

```python
# 1. GET DATA
df = pd.read_csv('data.csv')

# 2. EXPLORE (missing values, scales, distributions)
df.isnull().sum(); df.describe()

# 3. PIPELINE (scale -> pca -> cluster, all chained)
pipe = Pipeline([
    ('scale', StandardScaler()),
    ('pca', PCA(0.95)),
    ('kmeans', KMeans(n_clusters=k, n_init=10, random_state=0))
])

# 4. CHOOSE k (sweep, silhouette peak)
# 5. FIT & attach labels
df['cluster'] = pipe.fit_predict(df)

# ---- AFTER CLUSTERING ----
# 6. PROFILE   df.groupby('cluster').mean()
# 7. NAME      give each profile a human label
# 8. ACT       map each cluster to a decision/action
# 9. PREDICT   pipe.predict(new_data)  for incoming points
# 10. VALIDATE re-run with seeds, check ARI stability
```

**Why Pipeline is the key practice:** one object holds the whole transform→cluster chain, applies identical steps to new data, prevents order mistakes and data leakage. It's the difference between a notebook that works once and code you can actually deploy.

**The after-clustering arc:** evaluate → **profile → name → act → predict → validate**. Clustering produces a label; the value is in interpreting and using it.

---

# Your turn — build a real one

Pick a CSV from a domain you care about (Spotify features, your spending, sports stats, anything). Run this exact template on it:
1. Load it, explore it.
2. Pipeline: scale → (maybe PCA) → KMeans.
3. Choose k with silhouette.
4. Profile the clusters — what makes each group distinct?
5. Name them, decide an action per group.
6. Predict the segment for a made-up new row.
7. Check stability across seeds.

Recognizing whether YOUR clusters make sense is what makes the whole thing click.